In [4]:
import os
import pandas as pd

In [5]:
from pathlib import Path

In [6]:
BASE_DIR = Path.cwd()  # or Path(__file__).resolve().parent
data_path = BASE_DIR / "data" / "raw" / "combined_study_clinical_data.tsv"

nhl_df = pd.read_csv(data_path, sep="\t")

In [7]:
nhl_df.head()
nhl_df.info()
nhl_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2084 entries, 0 to 2083
Columns: 217 entries, Study ID to Year of Diagnosis
dtypes: float64(89), int64(1), object(127)
memory usage: 3.5+ MB


,Diagnosis Age,Age at Diagnosis,Neoplasm Disease Stage American Joint Committee on Cancer Code,Cllularity based on allele frequency,Aneuploidy Score,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Cluster,Last Communication Contact from Initial Pathologic Diagnosis Date,...,Time between excision and freezing,TMB (nonsynonymous),Total CNA,Total Rearrangement,Tumor Genome Coverage,Tumor in Normal Estimate,Tumor resected max dimension,Tumor Disease Anatomic Site,Patient Weight,Year of Diagnosis
count,414.000000,1078.000000,0.0,39.000000,47.000000,0.0,0.0,0.0,135.000000,86.000000,...,0.0,2010.000000,135.000000,135.000000,53.000000,107.000000,38.000000,0.0,96.000000,48.000000
mean,37.562802,61.393506,NaN,67.721332,6.340426,NaN,NaN,NaN,3.074074,1096.837209,...,NaN,1.659204,48.466667,1.185185,33.610377,0.003318,4.600526,NaN,71.039583,2009.145833
std,23.943429,14.958956,NaN,13.859046,6.380257,NaN,NaN,NaN,1.443615,1300.877938,...,NaN,5.905507,45.278706,1.270710,6.240367,0.005577,3.974216,NaN,18.110942,5.031770
min,1.000000,3.100000,NaN,46.127160,0.000000,NaN,NaN,NaN,0.000000,-10.000000,...,NaN,0.000000,6.000000,0.000000,24.810000,0.000000,0.400000,NaN,41.500000,1987.000000
25%,11.000000,53.225000,NaN,56.040013,1.000000,NaN,NaN,NaN,2.000000,133.750000,...,NaN,0.166667,19.000000,0.000000,29.880000,0.000000,2.000000,NaN,58.675000,2009.000000
50%,40.000000,63.100000,NaN,68.070829,5.000000,NaN,NaN,NaN,3.000000,739.500000,...,NaN,0.333333,35.000000,1.000000,32.180000,0.000000,3.000000,NaN,68.000000,2011.000000
75%,58.000000,72.000000,NaN,78.510704,9.000000,NaN,NaN,NaN,4.000000,1325.250000,...,NaN,1.200000,61.500000,2.000000,36.210000,0.005000,5.300000,NaN,81.925000,2012.000000
max,90.000000,93.400000,NaN,100.000000,24.000000,NaN,NaN,NaN,5.000000,5980.000000,...,NaN,142.900000,337.000000,6.000000,56.640000,0.040000,17.000000,NaN,127.700000,2013.000000


In [8]:
nhl_df.columns

Index(['Study ID', 'Patient ID', 'Sample ID', 'Diagnosis Age',
       'Age at Diagnosis', 'Age Category',
       'Neoplasm Disease Stage American Joint Committee on Cancer Code',
       'American Joint Committee on Cancer Publication Version Type',
       'Cllularity based on allele frequency', 'Aneuploidy Score',
       ...
       'Tumor in Normal Estimate', 'Tumor Purity',
       'Tumor resected max dimension', 'Person Neoplasm Status',
       'Tumor Disease Anatomic Site', 'Tumor Type', 'Vial number',
       'Patient's Vital Status', 'Patient Weight', 'Year of Diagnosis'],
      dtype='object', length=217)

In [9]:
missing = nhl_df.isna().mean().sort_values(ascending=False)
missing.head(25)

American Joint Committee on Cancer Tumor Stage Code                                 1.0
Adjuvant Postoperative Pharmaceutical Therapy Administered Indicator                1.0
Project code                                                                        1.0
Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code           1.0
Time between clamping and freezing                                                  1.0
Time between excision and freezing                                                  1.0
Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage    1.0
Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage          1.0
Stage Other                                                                         1.0
Specimen Freezing Means                                                             1.0
Specimen Current Weight                                                             1.0
Prior immunosuppressive therapy 

In [10]:
TARGET = "Overall Survival Status"

In [11]:
leakage_cols = [
    "Overall Survival (Months)",
    "Death from Initial Pathologic Diagnosis Date",
    "Last Communication Contact from Initial Pathologic Diagnosis Date",
    "Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value",
    "Progress Free Survival (Months)",
    "Months of disease-specific survival"
]

In [12]:
TARGET = "Overall Survival Status"

In [13]:


nhl_df[TARGET] = nhl_df[TARGET].map({
    "0:LIVING": 0,
    "1:DECEASED": 1
})

nhl_df[TARGET].value_counts()

Overall Survival Status
0.0    621
1.0    300
Name: count, dtype: int64

In [14]:
leakage_cols = [c for c in leakage_cols if c in nhl_df.columns]
nhl_df = nhl_df.drop(columns=leakage_cols)
print("Dropped:", leakage_cols)

Dropped: ['Overall Survival (Months)', 'Death from Initial Pathologic Diagnosis Date', 'Last Communication Contact from Initial Pathologic Diagnosis Date', 'Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value', 'Progress Free Survival (Months)', 'Months of disease-specific survival']


In [15]:
nhl_df = nhl_df.dropna(subset=[TARGET]).copy()
nhl_df.shape

(921, 211)

In [16]:
missing = nhl_df.isna().mean().sort_values(ascending=False)
missing.head(20)

Normal Genome Coverage                                                              1.0
Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage    1.0
Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage          1.0
Extranodal IPI Factor                                                               1.0
American Joint Committee on Cancer Tumor Stage Code                                 1.0
CNS Relapse                                                                         1.0
CNS Status                                                                          1.0
AnnArbor Stage IPI                                                                  1.0
Time between excision and freezing                                                  1.0
Time between clamping and freezing                                                  1.0
Prior immunosuppressive therapy other                                               1.0
Lymph node location positive pat

In [17]:
nhl_df[TARGET].value_counts(normalize=True)

Overall Survival Status
0.0    0.674267
1.0    0.325733
Name: proportion, dtype: float64

In [18]:
nhl_df.to_csv("data/processed/data_cleaned_for_early_detection.csv", index=False)

In [19]:
print("Saved: data_cleaned_for_early_detection.csv")

Saved: data_cleaned_for_early_detection.csv


In [20]:
id_cols = ["Patient ID", "Sample ID"]
id_cols = [c for c in id_cols if c in nhl_df.columns]

df = nhl_df.drop(columns=id_cols)

print("Dropped ID columns:", id_cols)

Dropped ID columns: ['Patient ID', 'Sample ID']


In [21]:
df.to_csv("data_cleaned_for_early_detection.csv", index=False)